# Section 6: Post-Metadata Dataset Audit and Visual Inspection
Audits true image validity, geometries, basic quality, and defines candidates for review. Does not alter/rewrite the operational manifest natively.


In [1]:
import pandas as pd
import numpy as np
import os
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

OUTPUT_ROOT = r"D:\GradProj\Skin Cancer Dataset\pipeline_output"
manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "training_eligible_manifest_post_metadata.csv")

d_audit = os.path.join(OUTPUT_ROOT, "audit_reports", "image_quality")
os.makedirs(d_audit, exist_ok=True)


## Load Post-Metadata Manifest & Validate Counts


In [2]:
print("Loading official post-metadata training eligible dataset...")
if not os.path.exists(manifest_path):
    raise FileNotFoundError(f"Manifest not found: {manifest_path}")

df = pd.read_csv(manifest_path)

total_rows = len(df)
class_counts = df["final_authoritative_label"].value_counts()

expected_total = 20513
expected_counts = {
    "NV": 12736,
    "MEL": 4468,
    "BCC": 3309
}

print("=== POST-METADATA DATASET STATE VALIDATION ===")
print(f"Total Rows: {total_rows} (Expected: {expected_total})")
print("Class Distributions:")
for cls, expected in expected_counts.items():
    actual = class_counts.get(cls, 0)
    print(f"{cls}: {actual} (Expected: {expected})")

count_errors = []
if total_rows != expected_total:
    count_errors.append(f"Total rows mismatch: expected {expected_total}, got {total_rows}")

for cls, expected in expected_counts.items():
    actual = class_counts.get(cls, 0)
    if actual != expected:
        count_errors.append(f"{cls} mismatch: expected {expected}, got {actual}")

if count_errors:
    raise ValueError(
        "Section 6 must run on the frozen post-metadata dataset state, but mismatches were found:\n- "
        + "\n- ".join(count_errors)
    )

print("\nDataset state matches the expected frozen post-metadata manifest.")
display(
    pd.DataFrame({
        "class_label": list(expected_counts.keys()),
        "count": [class_counts.get(c, 0) for c in expected_counts.keys()]
    })
)

Loading official post-metadata training eligible dataset...
=== POST-METADATA DATASET STATE VALIDATION ===
Total Rows: 20513 (Expected: 20513)
Class Distributions:
NV: 12736 (Expected: 12736)
MEL: 4468 (Expected: 4468)
BCC: 3309 (Expected: 3309)

Dataset state matches the expected frozen post-metadata manifest.


,class_label,count
0,NV,12736
1,MEL,4468
2,BCC,3309


## True Image Decode Validation & Basic Quality Extraction


In [3]:
def extract_image_specs(row):
    path = row["full_path"]
    specs = {
        "image_decode_success": False,
        "decode_error_message": "None",
        "decoded_width": 0,
        "decoded_height": 0,
        "decoded_channels": 0,
        "brightness_mean": np.nan,
        "dark_pixel_fraction": np.nan,
        "contrast_std": np.nan,
        "sharpness_laplacian_var": np.nan
    }

    try:
        img = cv2.imread(path)
        if img is None:
            specs["decode_error_message"] = "cv2.imread returned None (corrupt, missing, or unsupported)"
            return specs

        if len(img.shape) == 2:
            h, w = img.shape
            c = 1
            gray = img
        else:
            h, w, c = img.shape
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        brightness = float(gray.mean())
        dark_fraction = float((gray < 30).mean())
        contrast_std = float(gray.std())
        laplacian_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())

        specs.update({
            "image_decode_success": True,
            "decoded_width": w,
            "decoded_height": h,
            "decoded_channels": c,
            "brightness_mean": brightness,
            "dark_pixel_fraction": dark_fraction,
            "contrast_std": contrast_std,
            "sharpness_laplacian_var": laplacian_var
        })

    except Exception as e:
        specs["decode_error_message"] = str(e)

    return specs


print("Starting true image decode, geometry extraction, and basic quality metric extraction...")
tqdm.pandas(desc="Image Audit")

decoded_features = df.progress_apply(extract_image_specs, axis=1).tolist()
df_features = pd.DataFrame(decoded_features)

df_full = pd.concat([df.reset_index(drop=True), df_features.reset_index(drop=True)], axis=1)

# Create aspect ratio safely after decode extraction
df_full["aspect_ratio"] = df_full["decoded_width"] / df_full["decoded_height"].replace(0, 1)

print("=== DECODE SUMMARY ===")
print(f"Total rows audited: {len(df_full)}")
print(f"Decode successes: {(df_full['image_decode_success'] == True).sum()}")
print(f"Decode failures: {(df_full['image_decode_success'] == False).sum()}")

display(
    df_full[["image_decode_success", "decode_error_message"]]
    .groupby(["image_decode_success", "decode_error_message"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(10)
)

Starting true image decode, geometry extraction, and basic quality metric extraction...


Image Audit: 100%|██████████| 20513/20513 [08:37<00:00, 39.67it/s]

=== DECODE SUMMARY ===
Total rows audited: 20513
Decode successes: 20513
Decode failures: 0


,image_decode_success,decode_error_message,count
0,True,None,20513


## Decode & Geometry Summary


In [4]:
failed_decode = df_full[df_full["image_decode_success"] == False]
print(f"Decode Failures: {len(failed_decode)} files.")

df_valid = df_full[df_full["image_decode_success"] == True].copy()
df_valid["aspect_ratio"] = df_valid["decoded_width"] / df_valid["decoded_height"]

geo_summary = df_valid[["decoded_width", "decoded_height", "aspect_ratio"]].describe()
print("\n=== OVERALL GEOMETRY SUMMARY ===")
display(geo_summary)

print("\n=== CLASS GEOMETRY SUMMARY (MEAN / MEDIAN / MIN / MAX) ===")

class_geo_summary = (
    df_valid.groupby("final_authoritative_label")[["decoded_width", "decoded_height", "aspect_ratio"]]
    .agg(["mean", "median", "min", "max"])
)

display(class_geo_summary)

class_geo_summary.to_csv(
    os.path.join(d_audit, "image_dimension_summary_by_class.csv"),
    index=True
)

print("Saved: image_dimension_summary_by_class.csv")

Decode Failures: 0 files.

=== OVERALL GEOMETRY SUMMARY ===


,decoded_width,decoded_height,aspect_ratio
count,20513.000000,20513.000000,20513.000000
mean,850.254765,754.274606,1.185597
std,207.797729,270.164838,0.183621
min,576.000000,450.000000,0.750000
25%,600.000000,450.000000,1.000000
50%,1024.000000,768.000000,1.333333
75%,1024.000000,1024.000000,1.333333
max,1024.000000,1024.000000,1.546474



=== CLASS GEOMETRY SUMMARY (MEAN / MEDIAN / MIN / MAX) ===


decoded_width                    decoded_height  \
                                   mean  median  min   max           mean   
final_authoritative_label                                                   
BCC                          958.138410  1024.0  600  1024     934.838320   
MEL                          916.578782  1024.0  600  1024     844.288272   
NV                           798.957443   600.0  576  1024     675.783213   

                                             aspect_ratio                      \
                           median  min   max         mean    median       min   
final_authoritative_label                                                       
BCC                        1024.0  450  1024     1.051778  1.000000  1.000000   
MEL                        1024.0  450  1024     1.133113  1.000000  0.836914   
NV                          450.0  450  1024     1.238778  1.333333  0.750000   

                                     
                                max  
final_authoritative_label            
BCC                        1.333333  
MEL                        1.546474  
NV                         1.530643

Saved: image_dimension_summary_by_class.csv


## Quality Metrics Summary & Quality Candidate Generation


In [5]:
# =========================================================
# QUALITY METRICS SUMMARY & STRICTER QUALITY CANDIDATE GENERATION
# Replace the old candidate-generation cell with this
# =========================================================

# Summary metrics for valid decoded images
q_summary = df_valid[
    ["brightness_mean", "dark_pixel_fraction", "contrast_std", "sharpness_laplacian_var"]
].describe()

print("\n=== RAW QUALITY METRICS SUMMARY ===")
display(q_summary)

# -------------------------------------------------------------------
# Thresholds derived from the actual dataset distribution
# We use conservative outlier thresholds so the review queue stays small
# -------------------------------------------------------------------

# Geometry thresholds
width_low_pct = float(df_valid["decoded_width"].quantile(0.01))
height_low_pct = float(df_valid["decoded_height"].quantile(0.01))
ar_low_pct = float(df_valid["aspect_ratio"].quantile(0.005))
ar_high_pct = float(df_valid["aspect_ratio"].quantile(0.995))

# Quality thresholds
bright_low_pct = float(df_valid["brightness_mean"].quantile(0.01))
bright_high_pct = float(df_valid["brightness_mean"].quantile(0.99))
dark_high_pct = float(df_valid["dark_pixel_fraction"].quantile(0.99))
contrast_low_pct = float(df_valid["contrast_std"].quantile(0.02))
sharp_low_pct = float(df_valid["sharpness_laplacian_var"].quantile(0.02))

# More extreme thresholds for single-metric hard suspicion
contrast_very_low_pct = float(df_valid["contrast_std"].quantile(0.005))
sharp_very_low_pct = float(df_valid["sharpness_laplacian_var"].quantile(0.005))

threshold_table = pd.DataFrame([
    {"metric": "decoded_width_low_1pct", "threshold": width_low_pct},
    {"metric": "decoded_height_low_1pct", "threshold": height_low_pct},
    {"metric": "aspect_ratio_low_0.5pct", "threshold": ar_low_pct},
    {"metric": "aspect_ratio_high_99.5pct", "threshold": ar_high_pct},
    {"metric": "brightness_low_1pct", "threshold": bright_low_pct},
    {"metric": "brightness_high_99pct", "threshold": bright_high_pct},
    {"metric": "dark_pixel_fraction_high_99pct", "threshold": dark_high_pct},
    {"metric": "contrast_std_low_2pct", "threshold": contrast_low_pct},
    {"metric": "contrast_std_low_0.5pct", "threshold": contrast_very_low_pct},
    {"metric": "sharpness_low_2pct", "threshold": sharp_low_pct},
    {"metric": "sharpness_low_0.5pct", "threshold": sharp_very_low_pct},
])

print("\n=== DATA-DRIVEN QUALITY THRESHOLDS ===")
display(threshold_table)

threshold_table.to_csv(os.path.join(d_audit, "quality_flag_thresholds.csv"), index=False)

# -------------------------------------------------------------------
# Hard flags: strong reasons to inspect
# Soft flags: weaker signals that become meaningful in combination
# -------------------------------------------------------------------

cond_decode = df_full["image_decode_success"] == False

cond_small_dim = (
    (df_full["decoded_width"] < width_low_pct) |
    (df_full["decoded_height"] < height_low_pct)
)

cond_extreme_ar = (
    (df_full["image_decode_success"] == True) &
    (
        (df_full["aspect_ratio"] < ar_low_pct) |
        (df_full["aspect_ratio"] > ar_high_pct)
    )
)

cond_extreme_brightness = (
    (df_full["brightness_mean"] < bright_low_pct) |
    (df_full["brightness_mean"] > bright_high_pct)
)

cond_too_dark = df_full["dark_pixel_fraction"] > dark_high_pct
cond_low_contrast = df_full["contrast_std"] < contrast_low_pct
cond_very_low_contrast = df_full["contrast_std"] < contrast_very_low_pct
cond_low_sharpness = df_full["sharpness_laplacian_var"] < sharp_low_pct
cond_very_low_sharpness = df_full["sharpness_laplacian_var"] < sharp_very_low_pct

# -------------------------------------------------------------------
# Candidate logic
# Flag if:
# 1) any hard flag
# OR
# 2) two or more soft flags
# OR
# 3) a very extreme single soft flag
# -------------------------------------------------------------------

soft_flag_count = (
    cond_extreme_brightness.astype(int) +
    cond_too_dark.astype(int) +
    cond_low_contrast.astype(int) +
    cond_low_sharpness.astype(int)
)

candidate_mask = (
    cond_decode |
    cond_small_dim |
    cond_extreme_ar |
    (soft_flag_count >= 2) |
    cond_very_low_contrast |
    cond_very_low_sharpness
)

review_candidates = df_full[candidate_mask].copy()

def build_flag_reasons(row):
    reasons = []

    if row["image_decode_success"] == False:
        reasons.append("decode_failure")

    if row["decoded_width"] < width_low_pct or row["decoded_height"] < height_low_pct:
        reasons.append("small_dimensions")

    if row["image_decode_success"] == True:
        if row["aspect_ratio"] < ar_low_pct or row["aspect_ratio"] > ar_high_pct:
            reasons.append("extreme_aspect_ratio")

    if pd.notna(row["brightness_mean"]):
        if row["brightness_mean"] < bright_low_pct or row["brightness_mean"] > bright_high_pct:
            reasons.append("extreme_brightness")

    if pd.notna(row["dark_pixel_fraction"]) and row["dark_pixel_fraction"] > dark_high_pct:
        reasons.append("too_dark")

    if pd.notna(row["contrast_std"]):
        if row["contrast_std"] < contrast_very_low_pct:
            reasons.append("very_low_contrast")
        elif row["contrast_std"] < contrast_low_pct:
            reasons.append("low_contrast")

    if pd.notna(row["sharpness_laplacian_var"]):
        if row["sharpness_laplacian_var"] < sharp_very_low_pct:
            reasons.append("very_low_sharpness")
        elif row["sharpness_laplacian_var"] < sharp_low_pct:
            reasons.append("low_sharpness")

    return "|".join(reasons) if reasons else "none"

review_candidates["flag_reasons"] = review_candidates.apply(build_flag_reasons, axis=1)

review_candidates["soft_flag_count"] = soft_flag_count.loc[review_candidates.index]

print("\n=== STRICTER QUALITY REVIEW CANDIDATE SUMMARY ===")
print(f"Total quality review candidates: {len(review_candidates)}")
print(f"Candidate percentage: {round((len(review_candidates) / len(df_full)) * 100, 2)}%")

print("\n=== QUALITY REVIEW CANDIDATE COUNTS BY CLASS ===")
candidate_counts_by_class = (
    review_candidates["final_authoritative_label"]
    .value_counts()
    .rename_axis("class_label")
    .reset_index(name="candidate_count")
)
display(candidate_counts_by_class)

print("\n=== QUALITY REVIEW CANDIDATE COUNTS BY FLAG REASON ===")
flag_reason_counts = (
    review_candidates["flag_reasons"]
    .str.split("|", regex=False)
    .explode()
    .value_counts()
    .rename_axis("flag_reason")
    .reset_index(name="count")
)
display(flag_reason_counts)

# Save review outputs
review_candidates.to_csv(os.path.join(d_audit, "quality_review_candidates.csv"), index=False)
candidate_counts_by_class.to_csv(os.path.join(d_audit, "quality_candidate_counts_by_class.csv"), index=False)
flag_reason_counts.to_csv(os.path.join(d_audit, "quality_candidate_counts_by_reason.csv"), index=False)


=== RAW QUALITY METRICS SUMMARY ===


,brightness_mean,dark_pixel_fraction,contrast_std,sharpness_laplacian_var
count,20513.000000,20513.000000,20513.000000,20513.000000
mean,146.420805,0.059941,36.783200,65.265387
std,29.798365,0.149823,20.288529,124.356156
min,32.365700,0.000000,5.403519,1.973822
25%,132.664121,0.000000,21.479815,21.811211
50%,150.198045,0.000085,31.401118,36.078198
75%,166.434644,0.015407,46.699533,65.416462
max,239.581470,0.708912,105.918514,3825.024580



=== DATA-DRIVEN QUALITY THRESHOLDS ===


,metric,threshold
0,decoded_width_low_1pct,600.000000
1,decoded_height_low_1pct,450.000000
2,aspect_ratio_low_0.5pct,1.000000
3,aspect_ratio_high_99.5pct,1.508100
4,brightness_low_1pct,55.297780
5,brightness_high_99pct,203.455897
6,dark_pixel_fraction_high_99pct,0.667284
7,contrast_std_low_2pct,10.784860
8,contrast_std_low_0.5pct,8.605974
9,sharpness_low_2pct,9.231696



=== STRICTER QUALITY REVIEW CANDIDATE SUMMARY ===
Total quality review candidates: 458
Candidate percentage: 2.23%

=== QUALITY REVIEW CANDIDATE COUNTS BY CLASS ===


,class_label,candidate_count
0,NV,220
1,BCC,144
2,MEL,94



=== QUALITY REVIEW CANDIDATE COUNTS BY FLAG REASON ===


,flag_reason,count
0,extreme_brightness,147
1,too_dark,132
2,very_low_contrast,103
3,very_low_sharpness,103
4,extreme_aspect_ratio,91
5,low_sharpness,62
6,low_contrast,15
7,small_dimensions,5


In [6]:
print("=== SAMPLE QUALITY REVIEW CANDIDATES ===")
sample_cols = [
    "full_path",
    "final_authoritative_label",
    "decoded_width",
    "decoded_height",
    "aspect_ratio",
    "brightness_mean",
    "dark_pixel_fraction",
    "contrast_std",
    "sharpness_laplacian_var",
    "soft_flag_count",
    "flag_reasons"
]

available_sample_cols = [c for c in sample_cols if c in review_candidates.columns]
display(review_candidates[available_sample_cols].head(20))

=== SAMPLE QUALITY REVIEW CANDIDATES ===


,full_path,final_authoritative_label,decoded_width,decoded_height,aspect_ratio,brightness_mean,dark_pixel_fraction,contrast_std,sharpness_laplacian_var,soft_flag_count,flag_reasons
57,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000017...,NV,1024,673,1.521545,176.167650,0.000007,44.029472,321.138754,0,extreme_aspect_ratio
61,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000018...,NV,1024,669,1.530643,176.832897,0.000785,32.458113,364.601329,0,extreme_aspect_ratio
64,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000018...,NV,1024,675,1.517037,200.457551,0.000001,28.552264,43.855337,0,extreme_aspect_ratio
72,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000019...,NV,1024,672,1.523810,126.112848,0.001998,66.563184,367.438440,0,extreme_aspect_ratio
84,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000020...,NV,1024,675,1.517037,147.327636,0.000394,40.245646,244.903605,0,extreme_aspect_ratio
93,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000021...,NV,949,626,1.515974,169.360457,0.000000,55.135544,55.792742,0,extreme_aspect_ratio
100,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000022...,NV,959,634,1.512618,161.463120,0.000000,43.390062,86.594969,0,extreme_aspect_ratio
118,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000024...,NV,576,768,0.750000,160.582350,0.000000,36.671571,10.266219,0,small_dimensions|extreme_aspect_ratio
126,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000025...,NV,576,768,0.750000,136.798344,0.098432,63.282596,40.940179,0,small_dimensions|extreme_aspect_ratio
127,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000025...,NV,576,768,0.750000,140.935524,0.031261,37.408138,9.273266,0,small_dimensions|extreme_aspect_ratio


## Generate Class Sample & Unlabeled Grids


In [7]:
def generate_grid(sample_df, title, save_path, labels=True):
    if len(sample_df) == 0: return
    n = min(9, len(sample_df))
    sample_df = sample_df.sample(n, random_state=42)
    
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    fig.suptitle(title, fontsize=16)
    
    for ax, (_, row) in zip(axes.flatten(), sample_df.iterrows()):
        path = row["full_path"]
        if os.path.exists(path):
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        ax.axis("off")
        if labels:
            ax.set_title(row["final_authoritative_label"])
            
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

print("Building Visual Shortcut Grids...")
generate_grid(df_valid[df_valid["final_authoritative_label"]=="NV"], "Sample Grid: NV", os.path.join(d_audit, "class_sample_grid_NV.png"))
generate_grid(df_valid[df_valid["final_authoritative_label"]=="MEL"], "Sample Grid: MEL", os.path.join(d_audit, "class_sample_grid_MEL.png"))
generate_grid(df_valid[df_valid["final_authoritative_label"]=="BCC"], "Sample Grid: BCC", os.path.join(d_audit, "class_sample_grid_BCC.png"))
generate_grid(df_valid, "Mixed Unlabeled Audit Grid", os.path.join(d_audit, "mixed_unlabeled_sample_grid.png"), labels=False)
print("Grids securely built into Artifact directory.")


Building Visual Shortcut Grids...
Grids securely built into Artifact directory.


## Export Files & Checklists


In [8]:
# Save decode-level report
df_full.to_csv(os.path.join(d_audit, "image_decode_validation_report.csv"), index=False)

# Save geometry summary (overall)
geo_summary.rename_axis("statistic").to_csv(
    os.path.join(d_audit, "image_dimension_summary.csv"),
    index=True
)

# Save quality metric summary
q_summary.rename_axis("statistic").to_csv(
    os.path.join(d_audit, "image_quality_summary.csv"),
    index=True
)

# Save final dataset summary
summary_counts = pd.DataFrame([{
    "total_rows": total_rows,
    "decode_failures": len(failed_decode),
    "valid_decoded_rows": len(df_valid),
    "review_candidates": len(review_candidates),
    "NV_count": class_counts.get("NV", 0),
    "MEL_count": class_counts.get("MEL", 0),
    "BCC_count": class_counts.get("BCC", 0)
}])
summary_counts.to_csv(os.path.join(d_audit, "post_metadata_dataset_summary.csv"), index=False)

# Save artifact review note template
template = """ARTIFACT & SHORTCUT REVIEW TEMPLATE

1. Rulers / Scale Bars:
[Check whether rulers or scales appear and whether they are class-skewed]

2. Borders / Vignettes / Black Frames:
[Check whether border artifacts are common and class-specific]

3. Pen Markings / Surgical Marks:
[Check whether markings appear and whether they are class-skewed]

4. Text / Labels / Embedded Overlays:
[Check whether text or embedded overlays are present]

5. Crop Tightness / Framing:
[Check whether one class is systematically more tightly cropped]

6. Background / Style Bias:
[Check whether one class is photographed differently from the others]

7. Notes:
[Write any suspicious shortcut-learning patterns here]
"""
with open(os.path.join(d_audit, "artifact_review_notes.txt"), "w", encoding="utf-8") as f:
    f.write(template)

print("=== SECTION 6 FINAL SUMMARY ===")
print(f"Total post-metadata rows checked: {total_rows}")
print(f"Decode failures: {len(failed_decode)}")
print(f"Valid decoded rows: {len(df_valid)}")
print(f"Quality review candidates: {len(review_candidates)}")

print("\n=== FINAL CLASS COUNTS CHECKED IN SECTION 6 ===")
display(
    pd.DataFrame({
        "class_label": ["NV", "MEL", "BCC"],
        "count": [
            class_counts.get("NV", 0),
            class_counts.get("MEL", 0),
            class_counts.get("BCC", 0)
        ]
    })
)

print("\nSaved files:")
print("- post_metadata_dataset_summary.csv")
print("- image_decode_validation_report.csv")
print("- image_dimension_summary.csv")
print("- image_quality_summary.csv")
print("- quality_review_candidates.csv")
print("- class_sample_grid_NV.png")
print("- class_sample_grid_MEL.png")
print("- class_sample_grid_BCC.png")
print("- mixed_unlabeled_sample_grid.png")
print("- artifact_review_notes.txt")
print("- image_dimension_summary_by_class.csv")
print("- quality_flag_thresholds.csv")
print("- quality_candidate_counts_by_class.csv")
print("- quality_candidate_counts_by_reason.csv")

print("\nSection 6 completed as an audit-only stage. No rows were dropped and no raw images were rewritten.")

=== SECTION 6 FINAL SUMMARY ===
Total post-metadata rows checked: 20513
Decode failures: 0
Valid decoded rows: 20513
Quality review candidates: 458

=== FINAL CLASS COUNTS CHECKED IN SECTION 6 ===


,class_label,count
0,NV,12736
1,MEL,4468
2,BCC,3309



Saved files:
- post_metadata_dataset_summary.csv
- image_decode_validation_report.csv
- image_dimension_summary.csv
- image_quality_summary.csv
- quality_review_candidates.csv
- class_sample_grid_NV.png
- class_sample_grid_MEL.png
- class_sample_grid_BCC.png
- mixed_unlabeled_sample_grid.png
- artifact_review_notes.txt
- image_dimension_summary_by_class.csv
- quality_flag_thresholds.csv
- quality_candidate_counts_by_class.csv
- quality_candidate_counts_by_reason.csv

Section 6 completed as an audit-only stage. No rows were dropped and no raw images were rewritten.
